# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)



In [10]:
# EXERCISE 1
# convert to interger Total Claim Amount
df['Total Claim Amount'] = df['Total Claim Amount'].astype(int)

# Delete null values in Total Claim Amount (there are none)
df = df.dropna(subset=['Total Claim Amount'])

#filter df for total Claim amoun < 1000 & Response = 'Yes'
filtered_df = df[(df['Total Claim Amount'] < 1000) & (df['Response'] == 'Yes')]


In [ ]:
# Exercise 2
# For response 'Yes' and group by Policy Type and Gender the average monthly premium and customer lifetime value
grouped_df = (
	df[df['Response'] == 'Yes']
	.groupby(['Policy Type', 'Gender'])[['Monthly Premium Auto', 'Customer Lifetime Value','Total Claim Amount']]
	.mean()
	.reset_index()
)
grouped_df

# Insights for Total Claim Amount:
# 1. Customers with Personal Policy Type have a lower average Total Claim Amount compared to those with Corporate Policy Type.
# 2. Female customers tend to have a lower Total Claim Amount than male customers across both policy types.



,Policy Type,Gender,Monthly Premium Auto,Customer Lifetime Value,Total Claim Amount
0,Corporate Auto,F,94.301775,7712.628736,433.295858
1,Corporate Auto,M,92.188312,7944.465414,408.123377
2,Personal Auto,F,98.998148,8339.791842,452.498148
3,Personal Auto,M,91.085821,7448.383281,456.576493
4,Special Auto,F,92.314286,7691.584111,452.857143
5,Special Auto,M,86.343750,8247.088702,429.125000


In [ ]:
# EXERCISE 3. 

# Group by states with more than 500 customers

state_customer_counts = df['State'].value_counts()
filtered_states = state_customer_counts[state_customer_counts > 500]
filtered_states

# Insights:
# 1. The states with more than 500 customers are CA, WA, OR, AZ, and TX.
# 2. California (CA) has the highest number of customers, significantly more than any other state.
# 3. The distribution of customers across these states indicates a strong presence in the western region of the United States.

State
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64

In [ ]:
#Exercise 4
# Find the maximum, minimum, and median customer lifetime value by education level and gender
lifetime_value_stats = (
    df.groupby(['Education', 'Gender'])['Customer Lifetime Value']
    .agg(['max', 'min', 'median'])
    .reset_index()
)
lifetime_value_stats

#Conclusions:
# 1. Customers with a Doctorate degree tend to have the highest maximum Customer Lifetime Value across both genders.
# 2. The minimum Customer Lifetime Value is relatively consistent across different education levels, indicating a baseline value.
# 3. Median Customer Lifetime Value increases with higher education levels, suggesting that more educated customers may have a higher overall value to the company.
# 4. Gender differences are mixed across education levels — some groups show higher median CLV for females (e.g., Bachelor, Master) while others show higher median CLV for males (e.g., College, Doctor, High School); there is no consistent gender advantage overall.



,Education,Gender,max,min,median
0,Bachelor,F,73225.95652,1904.000852,5640.505303
1,Bachelor,M,67907.27050,1898.007675,5548.031892
2,College,F,61850.18803,1898.683686,5623.611187
3,College,M,61134.68307,1918.119700,6005.847375
4,Doctor,F,44856.11397,2395.570000,5332.462694
5,Doctor,M,32677.34284,2267.604038,5577.669457
6,High School or Below,F,55277.44589,2144.921535,6039.553187
7,High School or Below,M,83325.38119,1940.981221,6286.731006
8,Master,F,51016.06704,2417.777032,5729.855012
9,Master,M,50568.25912,2272.307310,5579.099207


In [32]:
#Exercise 5 
# The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

# First, ensure that the 'Effective To Date' column is in datetime format
df['Effective To Date'] = pd.to_datetime(df['Effective To Date'])
# Extract month from 'Effective To Date'
df['Month'] = df['Effective To Date'].dt.month_name()
# Create a pivot table with states as rows and months as columns
# The dataset column is named 'Number of Policies' — sum it to get total policies sold per state/month
policy_pivot = pd.pivot_table(
    df,
    index='State',
    columns='Month',
    values='Number of Policies',
    aggfunc='sum',
    fill_value=0
)
policy_pivot = policy_pivot.reindex(columns=[
    'January', 'February', 
])
policy_pivot  

Month,January,February
State,,
Arizona,3052,2864
California,5673,4929
Nevada,1493,1278
Oregon,4697,3969
Washington,1358,1225


In [39]:
#EXERCISE 6
#   Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold. Without using a pivot table and sorting from higher to lower.

# Get the top 3 states with the highest number of policies sold
top_states = df['State'].value_counts().head(3).index.tolist()
# Filter the DataFrame for these top states
top_states_df = df[df['State'].isin(top_states)]
# Group by State and Month, then count the number of policies sold
policies_by_state_month = (
    top_states_df.groupby(['State', 'Month'])['Number of Policies']
    .sum()
    .reset_index()
)
policies_by_state_month = policies_by_state_month.sort_values(by=['Number of Policies'], ascending=[False])
policies_by_state_month



,State,Month,Number of Policies
3,California,January,5673
2,California,February,4929
5,Oregon,January,4697
4,Oregon,February,3969
1,Arizona,January,3052
0,Arizona,February,2864


In [42]:
#EXERCISE 7
# The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

# Some datasets contain binary columns like 'Email', 'Phone', 'SMS', 'Mail' to indicate contact;
# this dataset does not. We'll first check for those columns and fallback to using 'Sales Channel'
# (which exists in this DataFrame) if the contact columns are not present.

contact_cols = [c for c in ['Email', 'Phone', 'SMS', 'Mail'] if c in df.columns]

if contact_cols:
    # Melt the DataFrame to unpivot marketing channels if those columns exist
    melted_df = pd.melt(
        df,
        id_vars=['Customer', 'Response'],
        value_vars=contact_cols,
        var_name='Marketing Channel',
        value_name='Contacted'
    )
    # Filter for contacted customers
    contacted_df = melted_df[melted_df['Contacted'] == 1]
    # Calculate response rate by marketing channel
    response_rate = (
        contacted_df.groupby('Marketing Channel')['Response']
        .apply(lambda x: (x == 'Yes').mean())
        .reset_index(name='Response Rate')
    )
else:
    # Fallback: use 'Sales Channel' column to compute response rate per channel
    response_rate = (
        df.groupby('Sales Channel')['Response']
        .apply(lambda x: (x == 'Yes').mean())
        .reset_index(name='Response Rate')
        .rename(columns={'Sales Channel': 'Marketing Channel'})
    )

response_rate

,Marketing Channel,Response Rate
0,Agent,0.180053
1,Branch,0.107876
2,Call Center,0.103223
3,Web,0.108856
